# Generic Utils
> This module handles all data/model/trainer loading logic based on the pipeline config. It provides a clean interface for the main training script, abstracting away the details of how different components are instantiated.

In [ ]:
#| default_exp utils.__init__

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore import *
from fastcore.utils import *

In [ ]:
#| export
from omegaconf import OmegaConf, DictConfig
import hydra
import torch

In [ ]:
#| export
def init_data(cfg: DictConfig):
    """Instantiates the correct datamodule based on the pipeline config."""
    print(f"Initializing Datamodule: {cfg.pipeline.datamodule._target_}")
    return hydra.utils.instantiate(cfg.pipeline.datamodule)


In [ ]:
#| export
def init_model(cfg: DictConfig):
    """
    Instantiates the model(s).
    Returns a single model for Stage 1, or a dictionary of 3 models for Stage 2.
    """
    stage = cfg.pipeline.stage
    model_cfg = cfg.pipeline.model

    if stage == 1:
        # Stage 1: Simply instantiate and return the single VQ-VAE model
        return hydra.utils.instantiate(model_cfg)

    elif stage == 2:
        # Stage 2: Instantiate all three models independently
        models = {
            "power_net": hydra.utils.instantiate(model_cfg.power_net),
            "jepa": hydra.utils.instantiate(model_cfg.jepa)
        }
        
        # Instantiate VQ-VAE, then immediately load its pretrained weights and freeze it
        vqvae = hydra.utils.instantiate(model_cfg.vqvae)
        
        if hasattr(model_cfg.vqvae, "checkpoint_path") and model_cfg.vqvae.checkpoint_path:
            print(f"Loading pretrained VQ-VAE weights from {model_cfg.vqvae.checkpoint_path}")
            checkpoint = torch.load(model_cfg.vqvae.checkpoint_path)
            vqvae.load_state_dict(checkpoint["state_dict"]) 
            
        # Freeze VQ-VAE because it's only used for getting latent codebooks
        vqvae.eval()
        for param in vqvae.parameters():
            param.requires_grad = False
            
        models["vqvae"] = vqvae
        return models
    
    else:
        raise ValueError(f"Unknown pipeline stage: {stage}")



In [ ]:
#| export
def init_trainer(cfg: DictConfig, data_module, models, device):
    """Instantiates the trainer and injects the loaded models and data."""
    # We pass models and datamodule directly into the instantiation call 
    # so the trainer receives its dependencies right away.
    return hydra.utils.instantiate(
        cfg.pipeline.trainer, 
        data_module=data_module,
        model=models, 
        device=device
    )


In [ ]:
# #| export
# def init_data(cfg):
#     """Initialize the data module based on the configuration. The configuration should specify the dataset name under `cfg.dataset.name`."""

#     data_modules_dict = {
#         "pov_dataset": VQDataModule,
#     }

#     data_name = cfg.dataset.name
#     assert data_name in data_modules_dict, f"Data module {data_name} not found. Available data modules: {list(data_modules_dict.keys())}"
#     data_module_cls = data_modules_dict[data_name]
#     data_module = data_module_cls(cfg)
#     return data_module


In [ ]:
# #| export
# def init_model(cfg, vqvae_ckp_path=None):
#     """Initialize the model based on the configuration.
#     The configuration should specify the model name under `cfg.model.name` and its parameters under `cfg.model`."""

#     model_config = OmegaConf.to_container(cfg.model, resolve=True)  
#     model_name = model_config.get(cfg.model.name)

#     if isinstance(model_name, str):
#         model_params = {k: v for k, v in model_config.items() if k != "name"}
#         model = VQVAE(**model_params)
#         return [model]

#     elif isinstance(model_name, list):
#         models_dct = { "VQVAE": VQVAE, "ARPredictor": ARPredictor, "PowerNet": PowerNet }
#         models = []
#         for name in model_name:
#             assert name in models_dct, f"Model {name} not found. Available models: {list(models_dct.keys())}"
#             model_cls = models_dct[name]
#             model_params = {k: v for k, v in model_config[name].items() if k != "name"}
#             model = model_cls(**model_params)
#             models.append(model)

#         if vqvae_ckp_path is not None:
#             vqvae_model = next((m for m in models if isinstance(m, VQVAE)), None)
#             if vqvae_model is not None:
#                 state_dict = torch.load(vqvae_ckp_path, map_location="cpu")["model_state_dict"]
#                 vqvae_model.load_state_dict(state_dict)
#                 print(f"Loaded VQVAE weights from {vqvae_ckp_path}")

#         return models

#     else:
#         raise ValueError(f"Invalid model name: {model_name}. Expected a string or a list of strings.")
    

In [ ]:
# #| export
# def init_trainer(cfg, model, device, slurm_jobid=None, **kwargs):
    
#     data_name = cfg.dataset.name
#     if data_name == "pov_dataset":
#         return VQVAETrainer(
#         cfg=cfg,
#         model=model,
#         device=device,
#         slurm_jobid=slurm_jobid
#     )

#     elif data_name == "another_dataset":
#         # Initialize a different trainer for another dataset
#         pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()